# Semenov et al. (2025), Figure 3C, from dandiset 001030

Claim. `fig3c-sustained-firing-2s` of `claims/001271/10.1016-j.xcrp.2025.102660.yaml`, a
`data: elsewhere` claim: the claims file assigns Figure 3C to
[DANDI:001030](https://dandiarchive.org/dandiset/001030/0.240520.1924) from its title. Version
0.240520.1924 is pinned: the paper cites no version, and it is the only published one.

The paper: Semenov I, Kim V, Silkuniene G, Pakhomov AG (2025), *Cell Reports Physical Science* 6,
102660, doi:10.1016/j.xcrp.2025.102660. The claims file was written from the NIH author
manuscript (PMC12320893) with its STAR Methods and supplement.

| Claim | Here |
|---|---|
| fig3c-sustained-firing-2s | tested |

The readings O1 to O4 of the claims file's header are run in every combination, with the whole
frame as a variant; the bleaching correction and which neurons are judged are run under the
readings named in the claim's section.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import re
import warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sinewave as sw

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 400)
DANDISET = "001030"
VERSION = sw.OTHER_DANDISETS[DANDISET]
ONSET_S, STIM_S, RAMP_S = 0.025, 2.0, 0.25   # dandiset description; Figure 3C legend (2 s, 250-ms ramp)
print("dandiset", DANDISET, "version", VERSION)

## Data check

In [ ]:
meta = sw.dandiset_metadata(DANDISET, VERSION)
print(meta["name"])
print(meta["description"])
print("related:", [(r.get("relation"), r.get("identifier")) for r in meta.get("relatedResource", [])])
assets = sw.list_assets(DANDISET, VERSION)
print(len(assets), "assets")


def load(a):
    f = sw._open(a["asset_id"])
    ts = f["acquisition/TimeLapseImaging"]
    d = ts["data"][:]
    desc = f["session_description"][()].decode()
    m = re.search(r"under ([\d.]+) ?V amplitude", desc)
    return dict(path=a["path"], subject=f["general/subject/subject_id"][()].decode(),
                session_id=f["general/session_id"][()].decode(), start=pd.Timestamp(f["session_start_time"][()].decode()),
                rate=float(ts["starting_time"].attrs["rate"]), n_frames=d.shape[0], shape=d.shape,
                volt=float(m.group(1)), cell=a["path"].split("-")[1], desc=desc,
                frame0_ratio=float(d[0].mean() / d[1:-1].mean()), last_ratio=float(d[-1].mean() / d[-2].mean()),
                alias=sw.alias_envelope(d, float(ts["starting_time"].attrs["rate"])), data=d)


with ThreadPoolExecutor(8) as ex:
    rec = list(ex.map(load, assets))
S = pd.DataFrame(rec).sort_values(["cell", "volt"]).reset_index(drop=True)

In [ ]:
print(S[["path", "subject", "session_id", "cell", "volt", "n_frames", "rate", "start"]].to_string(index=False))
print("\nshapes:", S["shape"].astype(str).value_counts().to_dict(), "| rates:", S["rate"].value_counts().to_dict())
print("solutions in descriptions:", S["desc"].str.contains("NaGluconate").sum(), "of", len(S), "name NaGluconate (0.8 S/m)")
print("sessions per cell:", S.groupby("cell").size().to_dict())
print("voltages per cell:", S.groupby("cell")["volt"].apply(lambda v: sorted(v)).to_dict())
print("subject identifiers distinct per file:", S["subject"].nunique(), "of", len(S))
print(f"frame 0 / mean of frames: {S['frame0_ratio'].min():.2f} to {S['frame0_ratio'].max():.2f}; "
      f"last frame / the one before: {S['last_ratio'].min():.3f} to {S['last_ratio'].max():.3f}")
for _, r in S.iterrows():
    on = int(round(ONSET_S * r["rate"]))
    off = int(round((ONSET_S + STIM_S) * r["rate"]))
    print(f"{r['cell']} {r['volt']:.2f} V: {r['n_frames']} frames = {1000 * r['n_frames'] / r['rate']:.0f} ms; "
          f"stimulus frames {on} to {off}{' (recording ends before the stimulus ends)' if off > r['n_frames'] else ''}")

Cell identity: the mean frames of sessions named for the same cell against those of other cells.
Stimulus timing: the aliased 2-kHz stimulus (|2000 - 3134.8| = 1,134.8 Hz) in the images.

In [ ]:
mp = np.vstack([r["data"][1:70].mean(0).ravel() for _, r in S.iterrows()])
cm = np.corrcoef(mp)
same = S["cell"].to_numpy(dtype=object)[:, None] == S["cell"].to_numpy(dtype=object)[None, :]
iu = np.triu_indices(len(S), 1)
print(f"mean-frame correlation, same cell: min {cm[iu][same[iu]].min():.3f}; different cells: max {cm[iu][~same[iu]].max():.3f}")
rows = []
for _, r in S.iterrows():
    env = r["alias"]
    on = int(round(ONSET_S * r["rate"]))
    base = env[20:on - 10].mean()
    late = env[int(1.0 * r["rate"]):int(1.9 * r["rate"])].mean()
    thr = base + 0.5 * (late - base)
    above = np.where(env[20:len(env) - 20] > thr)[0] + 20
    rows.append(dict(cell=r["cell"], volt=r["volt"], ratio=late / base,
                     last_above_ms=1000 * above.max() / r["rate"] if len(above) else np.nan))
al = pd.DataFrame(rows)
print(al.round(2).to_string(index=False))
print("expected stimulus end: 2025 ms (onset 25 ms + 2 s)")

## fig3c-sustained-firing-2s

> In the low-conductivity solution, a 2-s unmodulated 2-kHz sine wave ramped over 250 ms
> evoked firing for its whole duration, rising with field until replaced by a firing block with
> sustained depolarization.

Criterion: traces corrected for bleaching by subtracting a trace without APs (Methods), AP
readings of the header. Reproduced when an intermediate field has APs in the last 250 ms of the
stimulus, the AP count rises with field up to some field, and the highest field shows sustained
depolarization (O4, over the last 250 ms of the stimulus).

Details taken here, each a choice the criterion leaves open:
- The trace without APs is the neuron's own session with no AP under the reading. Only cell c7
  has one (1.0 V). For the others, readings: (a) no correction, (b) subtract the dataset's only
  AP-free trace, c7 at 1.0 V. The reference session is not itself judged.
- "Rises with field up to some field": the AP count at the field with the most APs is larger than
  at the lowest field, and that field is not the lowest (a standard choice).
- Which neurons: Figure 3C shows one neuron. Readings: (a) at least one neuron meets all three
  parts, (b) every neuron with at least three fields does.
- O4(b) uses the median AP height of all the neuron's sessions (there is no change of solution).
- APs are counted during the stimulus (25 to 2,025 ms, or the end of the recording).

In [ ]:
def roi_trace(d, roi):
    on = int(round(ONSET_S * 3134.8))
    m = sw.soma_masks(d[1:on].astype(float).mean(0))[roi]
    if not m.any():
        return None
    tr = d[:, m].astype(float).mean(1)
    tr[0] = np.nan
    tr[-1] = np.nan
    return tr


def run(roi, f0r, apr, sdr, bleach):
    out = []
    xs = {}
    for i, r in S.iterrows():
        tr = roi_trace(r["data"], roi)
        if tr is None:
            continue
        rate = r["rate"]
        on = int(round(ONSET_S * rate))
        lo = 1 if f0r == "all" else on - int(round(0.020 * rate))
        f0 = np.nanmean(tr[lo:on])
        xs[i] = (100 * (tr - f0) / f0, lo, on)
    # APs on uncorrected traces first, to find each cell's AP-free session
    raw_aps = {}
    for i, (x, lo, on) in xs.items():
        r = S.loc[i]
        off = min(int(round((ONSET_S + STIM_S) * r["rate"])), len(x) - 1)
        sd = np.nanstd(x[lo:on], ddof=1)
        lvl = 5.0 if apr == "5pct" else 5 * sd
        raw_aps[i] = len(sw.detect_aps(x, r["rate"], lvl, on, off))
    ref = {}
    for cell, g in S.groupby("cell"):
        free = [i for i in g.index if i in xs and raw_aps[i] == 0]
        ref[cell] = min(free, key=lambda i: S.loc[i, "volt"]) if free else None
    c7_ref = ref.get("c7")
    for i, (x, lo, on) in xs.items():
        r = S.loc[i]
        rate = r["rate"]
        use = ref[r["cell"]] if ref[r["cell"]] is not None else (c7_ref if bleach == "c7" else None)
        if use is not None and use != i and use in xs:
            xr = xs[use][0]
            n = min(len(x), len(xr))
            y = x[:n] - xr[:n]
        else:
            y = x.copy()
        off = min(int(round((ONSET_S + STIM_S) * rate)), len(y) - 1)
        w0 = int(round((ONSET_S + STIM_S - 0.25) * rate))
        pre_sd = np.nanstd(y[lo:on], ddof=1)
        lvl = 5.0 if apr == "5pct" else 5 * pre_sd
        aps = sw.detect_aps(y, rate, lvl, on, off)
        out.append(dict(i=i, cell=r["cell"], volt=r["volt"], is_ref=(use == i) or (ref[r["cell"]] == i),
                        corrected_by=None if use is None or use == i else f"{S.loc[use, 'cell']} {S.loc[use, 'volt']:g} V",
                        n_ap=len(aps), ap_last250=int((aps["frame"] >= w0).sum()),
                        last250_mean=float(np.nanmean(y[w0:off])), pre_sd=pre_sd,
                        peaks=aps["peak"].to_numpy(), truncated=off < int(round((ONSET_S + STIM_S) * rate)), y=y))
    o = pd.DataFrame(out)
    med_h = o.groupby("cell")["peaks"].apply(lambda v: np.median(np.concatenate(list(v))) if sum(map(len, v)) else np.nan)
    o["sd_level"] = 5 * o["pre_sd"] if sdr == "5sd" else 0.5 * o["cell"].map(med_h)
    o["sustained"] = (o["ap_last250"] == 0) & (o["last250_mean"] > o["sd_level"])
    return o


def judge(o):
    rows = []
    for cell, g in o[~o["is_ref"]].groupby("cell"):
        g = g.sort_values("volt")
        if len(g) < 2:
            rows.append(dict(cell=cell, fields=len(g), inter_ap_last250=False, rises=False, top_sd=False))
            continue
        inter = g.iloc[1:-1] if len(g) >= 3 else g.iloc[0:0]
        peak_i = g["n_ap"].to_numpy().argmax()
        rows.append(dict(cell=cell, fields=len(g), inter_ap_last250=bool((inter["ap_last250"] > 0).any()),
                         rises=bool(peak_i > 0 and g["n_ap"].iloc[peak_i] > g["n_ap"].iloc[0]),
                         top_sd=bool(g["sustained"].iloc[-1])))
    j = pd.DataFrame(rows)
    j["all3"] = j["inter_ap_last250"] & j["rises"] & j["top_sd"]
    return j


res, store = [], {}
for roi in ("otsu", "otsu_eroded", "frame"):
    for f0r in ("all", "last20"):
        for apr in ("5pct", "5sd"):
            for sdr in ("5sd", "half_ap"):
                for bleach in ("none", "c7"):
                    o = run(roi, f0r, apr, sdr, bleach)
                    j = judge(o)
                    store[(roi, f0r, apr, sdr, bleach)] = (o, j)
                    elig = j[j["fields"] >= 3]
                    res.append(dict(roi=roi, f0=f0r, ap=apr, sdr=sdr, bleach=bleach, cells=len(j),
                                    inter_ap_last250=int(j["inter_ap_last250"].sum()), rises=int(j["rises"].sum()),
                                    top_sd=int(j["top_sd"].sum()), c7_inter_and_rises=bool(j.loc[j["cell"] == "c7", "inter_ap_last250"].any() and j.loc[j["cell"] == "c7", "rises"].any()),
                                    cells_all3=int(j["all3"].sum()), cells_3fields=len(elig),
                                    cells_3fields_all3=int(elig["all3"].sum()),
                                    passing=",".join(j.loc[j["all3"], "cell"]),
                                    any_cell="reproduced" if j["all3"].any() else "not reproduced",
                                    every_cell="reproduced" if len(elig) and elig["all3"].all() else "not reproduced"))
R = pd.DataFrame(res)
print(R.to_string(index=False))
claims_roi = R[R["roi"].isin(["otsu", "otsu_eroded"])]
print("\nclaims-file ROIs, reading (a) at least one neuron:", claims_roi["any_cell"].value_counts().to_dict())
print("claims-file ROIs, reading (b) every neuron with 3 or more fields:", claims_roi["every_cell"].value_counts().to_dict())

In [ ]:
for key in [("otsu", "all", "5pct", "5sd", "none"), ("otsu", "last20", "5sd", "half_ap", "c7"), ("frame", "all", "5pct", "5sd", "none")]:
    o, j = store[key]
    print(key)
    print(o[["cell", "volt", "is_ref", "corrected_by", "n_ap", "ap_last250", "last250_mean", "pre_sd", "sd_level", "sustained", "truncated"]]
          .round(2).to_string(index=False))
    print(j.to_string(index=False))
    print()

In [ ]:
o, j = store[("frame", "all", "5pct", "5sd", "none")]
fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True)
for ax, (cell, g) in zip(axes, o.groupby("cell")):
    for k, (_, r) in enumerate(g.sort_values("volt").iterrows()):
        t = np.arange(len(r["y"])) / 3134.8
        ax.plot(t, r["y"] + 20 * k, lw=0.3)
        ax.text(2.25, 20 * k, f"{r['volt']:g} V", fontsize=7)
    ax.set_title(f"{cell} (whole frame, uncorrected)", fontsize=8)
    ax.axvspan(ONSET_S, ONSET_S + STIM_S, color="0.92", zorder=0)
    ax.set_xlabel("s")
axes[0].set_ylabel("dF (%), offset 20% per voltage")
plt.tight_layout()
plt.show()
ally = np.concatenate([np.asarray(r["y"])[1:-1] for _, r in o.iterrows()])
print(f"plotted dF range {np.nanmin(ally):.1f} to {np.nanmax(ally):.1f}% before offsets")

In [ ]:
rows = []
for key, (o, j) in store.items():
    r = o[(o["cell"] == "c7") & (o["volt"] == 2.3)]
    if len(r):
        r = r.iloc[0]
        rows.append(dict(zip(["roi", "f0", "ap", "sdr", "bleach"], key), n_ap=r["n_ap"], ap_last250=r["ap_last250"],
                         last250_mean=round(r["last250_mean"], 2), sd_level=round(r["sd_level"], 2),
                         peaks_last250=np.round(r["peaks"][-r["ap_last250"]:], 1).tolist() if r["ap_last250"] else []))
c7 = pd.DataFrame(rows)
print("c7 at 2.3 V, the highest field of the cell with the most fields:")
print(c7.to_string(index=False))
print(f"\nmean dF over the last 250 ms above the O4 level in {int((c7['last250_mean'] > c7['sd_level']).sum())} of {len(c7)} "
      f"readings; no AP in that window in {int((c7['ap_last250'] == 0).sum())} of {len(c7)}")

**Result: partially reproduced.** The first two parts hold with the O1(a) ROI: under every F0,
height and bleaching reading, c7 (the cell with six fields) has APs in the last 250 ms of the
2-s wave at an intermediate field and an AP count that rises from its lowest field; with the 5%
height all four cells do, and with the 5-SD height 1 to 3 cells meet each part. The O1(b) ROI
leaves two cells (not c7), and with its 5-SD height neither meets the first part. The third part
fails everywhere: under no reading, in the claims file's ROIs or the whole frame, does any
cell's highest field show sustained depolarization (O4), so the criterion is not met for at
least one cell (reading a) or for every cell (reading b). In c7,
whose fields reach 2.3 V, firing drops at the top field (from 70 to 90 APs at 1.45 to 2.0 V to
16 at 2.3 V with O1(a) and the 5% height, 20 in the whole frame). With the O1(a) ROI its mean dF
over the last 250 ms at 2.3 V (1.7 to 7.1%) stays below the O4 levels (6.5 to 14.5%); in the
whole-frame variant it is 6.0%, above the 5-SD level (4.5 to 4.7%), but one 8.0% peak in that
window counts as an AP, which O4 excludes. The c5 recordings at 1.5 and 2.0 V end at 1,914 ms, before the stimulus
ends (2,025 ms); their last-250-ms window is cut to 139 ms.

## Summary

| Claim | Outcome |
|---|---|
| fig3c-sustained-firing-2s | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

The analysis follows the claims file's readings; the trace without APs used for bleaching is the
neuron's own AP-free session where there is one (c7 only), and the reading for the other cells is
either none or c7's. The sustained depolarization measure (O4) is the claims file's.

About DANDI:001030: it holds 17 sessions of 4 cells (c5 to c8) in NaGlu solution, each file its
own subject named by cell and voltage ("c7_1,0V" is cell 7 at 1.0 V), with the voltage measured
as one polarity; two c5 recordings end at 1,914 ms, before the 2-s stimulus ends; the dandiset
does not name the paper.